# 2 · Hyperparameter search

Finds good settings for the three model families on the annotated series 1.

* The window features are computed **once** per sausage (`ml_core.training.load_windows`). Windowing is
  stateless, so this gives the same result as windowing inside every fold, but much faster.
* Successive halving grid search (`HalvingGridSearchCV`) with **leave-one-sausage-out** cross-validation,
  scored by **balanced accuracy**, because the phases have very different lengths.
* The winners are written to `data/models/hyperparameters.json`. Notebook 3 reads them from there.
* `FEATURE_SET` selects which sensor signals the models may use (`FEATURE_SETS` in `ml_core/training.py`). The default,
  `temperature_difference`, uses the probe temperatures relative to the air instead of absolute temperatures: in the only
  annotated run (spring → summer) the absolute temperature follows the season (see notebook 04). Search with the same
  feature set you train with in notebook 3.
* Search space: `PARAM_GRIDS` in `ml_core/training.py`. `QUICK = True` runs tiny grids, for testing the notebook only.

Runtime on the full data: tens of minutes on CPU. If RAPIDS is installed (GPU), the next cell switches it on.

In [1]:
# Optional GPU acceleration. Must run before scikit-learn is imported.
try:
    import cuml
    cuml.accel.install()
    GPU = True
except Exception as e:  # not installed or no GPU: plain scikit-learn on CPU
    GPU = False
print("RAPIDS cuML acceleration:", GPU)

RAPIDS cuML acceleration: False


In [2]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # project root, so ml_core is importable
import pandas as pd
from ml_core import data
from ml_core.training import (DEFAULT_FEATURE_SET, MODEL_NAMES, PARAM_GRIDS, QUICK_GRIDS, load_windows,
                              save_hyperparameters, search_hyperparameters)
pd.set_option("display.max_colwidth", 200)

In [3]:
CONVERTED_DIR = data.CONVERTED_DIR
MODELS_DIR = data.MODELS_DIR
WINDOW_SIZE, STRIDE = 10, 2
FEATURE_SET = DEFAULT_FEATURE_SET   # see FEATURE_SETS in ml_core/training.py
QUICK = False
N_JOBS = 1 if GPU else -1

In [4]:
windows = load_windows(CONVERTED_DIR, window_size=WINDOW_SIZE, stride=STRIDE).select(FEATURE_SET)
print(f"Feature set {FEATURE_SET}: {len(windows.columns)} features per window")
windows.summary()

series_1_sausage_1: 41,462 rows → 20,727 windows


series_1_sausage_2: 41,479 rows → 20,735 windows
series_1_sausage_3: 41,468 rows → 20,730 windows


series_1_sausage_4: 41,472 rows → 20,732 windows
series_1_sausage_5: 41,465 rows → 20,728 windows
Feature set temperature_difference: 24 features per window


,recording,rows,windows,from,to,Umroetung,Reifung Phase 1,Reifung Phase 2
0,series_1_sausage_1,41462,20727,2023-03-30 00:33:38.051909+00:00,2023-08-02 23:56:34.268438+00:00,7304,8116,5307
1,series_1_sausage_2,41479,20735,2023-03-30 00:33:38.051909+00:00,2023-08-02 23:43:26.013334+00:00,7309,8120,5306
2,series_1_sausage_3,41468,20730,2023-03-30 00:38:00.000000+00:00,2023-08-02 23:58:00.000000+00:00,7307,8118,5305
3,series_1_sausage_4,41472,20732,2023-03-30 00:37:55.818346+00:00,2023-08-02 23:55:25.481812+00:00,7308,8120,5304
4,series_1_sausage_5,41465,20728,2023-03-30 00:33:38.051909+00:00,2023-08-02 23:49:39.000000+00:00,7307,8117,5304


In [5]:
result = search_hyperparameters(windows, grids=QUICK_GRIDS if QUICK else PARAM_GRIDS, n_jobs=N_JOBS)


=== Logistic Regression ===  (feature set: temperature_difference)


n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 2073
max_resources_: 103652
aggressive_elimination: False
factor: 4
----------
iter: 0
n_candidates: 27
n_resources: 2073
Fitting 5 folds for each of 27 candidates, totalling 135 fits


----------
iter: 1
n_candidates: 7
n_resources: 8292
Fitting 5 folds for each of 7 candidates, totalling 35 fits


----------
iter: 2
n_candidates: 2
n_resources: 33168
Fitting 5 folds for each of 2 candidates, totalling 10 fits


best balanced_accuracy = 0.713 in 16.3 min: {'C': 100, 'class_weight': 'balanced', 'l1_ratio': 0, 'max_iter': 1000, 'solver': 'liblinear'}

=== Random Forest ===  (feature set: temperature_difference)
n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 2073
max_resources_: 103652
aggressive_elimination: False
factor: 4
----------
iter: 0
n_candidates: 32
n_resources: 2073
Fitting 5 folds for each of 32 candidates, totalling 160 fits


----------
iter: 1
n_candidates: 8
n_resources: 8292
Fitting 5 folds for each of 8 candidates, totalling 40 fits


----------
iter: 2
n_candidates: 2
n_resources: 33168
Fitting 5 folds for each of 2 candidates, totalling 10 fits


best balanced_accuracy = 0.737 in 1.1 min: {'bootstrap': True, 'class_weight': 'balanced_subsample', 'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 200}

=== MLP (neural net) ===  (feature set: temperature_difference)
n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 2073
max_resources_: 103652
aggressive_elimination: False
factor: 4
----------
iter: 0
n_candidates: 32
n_resources: 2073
Fitting 5 folds for each of 32 candidates, totalling 160 fits


----------
iter: 1
n_candidates: 8
n_resources: 8292
Fitting 5 folds for each of 8 candidates, totalling 40 fits


----------
iter: 2
n_candidates: 2
n_resources: 33168
Fitting 5 folds for each of 2 candidates, totalling 10 fits


best balanced_accuracy = 0.735 in 2.8 min: {'activation': 'tanh', 'alpha': 0.001, 'batch_size': 32, 'early_stopping': True, 'hidden_layer_sizes': [64, 32], 'learning_rate_init': 0.0003, 'max_iter': 500, 'validation_fraction': 0.15}


## Best candidates per model (last halving round)

In [6]:
for key, r in result.items():
    print(f"{MODEL_NAMES[key]}: best balanced accuracy {r['score']:.3f}")
    display(r["results"].head(5))

Logistic Regression: best balanced accuracy 0.713


,C,class_weight,l1_ratio,max_iter,solver,balanced_accuracy,std
0,100,balanced,0,1000,liblinear,0.713,0.083
1,100,balanced,1,1000,liblinear,0.708,0.079


Random Forest: best balanced accuracy 0.737


,bootstrap,class_weight,criterion,max_depth,max_features,min_samples_leaf,min_samples_split,n_estimators,balanced_accuracy,std
0,True,balanced_subsample,gini,10,sqrt,4,2,200,0.737,0.089
1,True,balanced_subsample,gini,10,sqrt,4,2,500,0.731,0.086


MLP (neural net): best balanced accuracy 0.735


,activation,alpha,batch_size,early_stopping,hidden_layer_sizes,learning_rate_init,max_iter,validation_fraction,balanced_accuracy,std
0,tanh,0.0010,32,True,"(64, 32)",0.0003,500,0.15,0.735,0.086
1,tanh,0.0001,32,True,"(64, 32)",0.0003,500,0.15,0.734,0.086


In [7]:
path = save_hyperparameters(result, windows, MODELS_DIR)
print("Saved", path)

Saved F:\_Data\Uni\MA_Project\masterproject-lorawan-demonstrator\data\models\hyperparameters.json
